# PD-L1 Detection 폴더 평가

1. `TEST_FOLDER`에 평가 폴더를 지정합니다(하위 폴더까지 전체 탐색).
2. 전체 실행하면 설정된 모델의 Dice, mAP@0.5, 패치 CPS Grade PCC를 비교합니다.

이미지와 같은 이름의 JSON 라벨이 필요합니다. `cells: []`인 빈 GT도 포함합니다.
모든 이미지를 **1024×1024로 리사이즈 → 512×512 네 조각(좌상·우상·좌하·우하)**으로 평가합니다.
랜덤 크롭, 신규 12장 선택, GT 개수 필터, other 제외 비교는 사용하지 않습니다.
기존 라벨 로더의 비정상 bbox 처리는 유지합니다.

- **Dice**: other 포함 전체 사각형의 합집합 영역을 비교하고 모든 패치의 면적을 합산합니다. 클래스 무관 점수입니다.
- **mAP@0.5**: 5개 클래스의 AP 평균(101-point 보간). GT 없는 패치의 예측도 FP로 반영합니다.
- **PCC**: 각 512×512 패치의 CPS Grade 상관계수. Grade 경계는 `<1 / 1–<5 / 5–<10 / ≥10`입니다.
  CPS = `min(100, 100*(Positive Epithelial+Positive Immune)/(Negative Epithelial+Positive Epithelial))`.
  분모가 0이면 CPS는 NaN으로 기록하며 PCC 유효 쌍 수를 표시합니다.

빈 패치를 데이터에서 제거하지 않습니다. Dice의 양쪽 영역이 모두 0이면 패치 점수는 NaN,
전체에 GT가 없는 클래스의 AP와 상수/표본 부족 PCC도 정의할 수 없어 NaN입니다.
아래 셀에 `load_test_folder`, `predict_patches`, `calculate_dice`, `calculate_map`, `calculate_pcc`를 직접 구현했습니다.

클래스별 Dice·AP@0.5와 전체 통합 표를 함께 표시합니다. `dice_class_mean`은 클래스별 pooled Dice의 산술평균(NaN 제외), `dice_union`은 클래스를 구분하지 않은 전체 영역 Dice입니다. mAP는 클래스 AP의 평균이며 CPS Grade PCC는 전체 표에만 표시합니다.

평가 함수 `calculate_dice`, `calculate_map`, `calculate_pcc`는 노트북에 두고, 보조 함수는 `detection_support/helpers.py`에 분리했습니다.

예측 결과는 모델별 `*_patches/patch_*.npz`에 즉시 저장하고, 지표 계산 시 패치 단위로 다시 읽습니다.
이미지나 전체 예측 배열을 RAM에 누적하지 않습니다. 정확한 AP 계산의 confidence 정렬에는
클래스별 예측 수에 비례하는 메모리가 여전히 필요합니다. `SAVE_PREDICTIONS_JSON=True`이면
JSON도 패치 단위로 저장합니다. 임계값과 지표 정의는 변경하지 않았습니다.


In [ ]:
from tqdm.auto import tqdm
from detection_support import model as nn
from detection_support import ops as util
from detection_support.dataset import IMAGE_SUFFIXES, FourCropDataset
from detection_support.helpers import (
    clip_boxes, rectangle_dice, average_precision_50,
    cps_from_counts, cps_grade, paired_score_correlation,
    PredictionStore, save_predictions_json,
)
import gc
import zipfile
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import numpy as np
import pandas as pd
import torch
from IPython.display import display

# 모델 출력 ID 순서와 일치해야 합니다. CPS도 이 5개 클래스 순서를 사용합니다.
CLASS_NAMES = {
    0: 'Negative Epithelial',
    1: 'Negative Immune',
    2: 'Positive Epithelial',
    3: 'Positive Immune',
    4: 'other',
}
# JSON에서 읽을 이름: 선언되지 않은 이름(Tumor 등)은 세포 GT에 포함하지 않습니다.
LABEL_ALIASES = {
    0: ['Negative Epithelial', 'Negative Tumor(Epithelial)'],
    1: ['Negative Immune', 'Negative Lymphocyte', 'Negative Macrophage'],
    2: ['Positive Epithelial', 'Positive Tumor(Epithelial)'],
    3: ['Positive Immune', 'Positive Lymphocyte', 'Positive Macrophage'],
    4: ['other'],
}

TEST_FOLDER = Path('../../../data/mediautoAI_PDL1_Stomach')  # 이 폴더만 변경하세요
MODEL_ROOT = Path('../../../model/mediautoAI_PDL1_Stomach/pdl1_flat_5class/last_model.pt')
CHECKPOINTS = {
    'baseline': MODEL_ROOT,

}
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
SAVE_PREDICTIONS_JSON = False  # NPZ는 항상 저장; JSON이 필요한 경우만 True
AP_CONFIDENCE = 0.001
CONFIDENCE = 0.10  # Dice 및 CPS에 사용
NMS_IOU = 0.45
OUTPUT_DIR = Path('results') / ('detection_test_' + datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f'))


In [ ]:
class EvaluationDataset(FourCropDataset):
    """Read only explicitly configured cell classes; preserve every source image."""
    def __init__(self, records, class_names, label_aliases, **kwargs):
        self.class_names = dict(class_names)
        self.label_to_id = {}
        if set(label_aliases) - set(class_names):
            raise ValueError('LABEL_ALIASES contains IDs absent from CLASS_NAMES')
        for class_id, name in class_names.items():
            for alias in [name, *label_aliases.get(class_id, [])]:
                key = alias.strip().casefold()
                if key in self.label_to_id and self.label_to_id[key] != class_id:
                    raise ValueError(f'Ambiguous label alias: {alias}')
                self.label_to_id[key] = class_id
        super().__init__(records, **kwargs)

    def load_labels(self, label_path, image_size=None):
        if image_size is None:
            raise ValueError('image_size is required')
        annotations = json.loads(Path(label_path).read_text(encoding='utf-8'))
        if not isinstance(annotations, dict) or not isinstance(annotations.get('cells'), list):
            raise ValueError(f'{label_path}: expected a cells list')
        width, height = image_size
        labels = []
        for cell in annotations['cells']:
            class_id = self.label_to_id.get(str(cell.get('class_name', '')).strip().casefold())
            if class_id is None:
                continue
            box = self._mediauto_box(cell, width, height)
            if box is None:
                continue
            x0, y0, x1, y1 = box
            labels.append([class_id, (x0+x1)/(2*width), (y0+y1)/(2*height),
                           (x1-x0)/width, (y1-y0)/height])
        return np.asarray(labels, dtype=np.float32).reshape(-1, 5)


def load_test_folder(folder):
    """Read every image recursively; require a sibling JSON, including empty cells lists."""
    folder = Path(folder)
    if not folder.is_dir():
        raise NotADirectoryError(folder)
    images = sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
    if not images:
        raise ValueError(f'No images in {folder}')
    missing = [str(p) for p in images if not p.with_suffix('.json').is_file()]
    if missing:
        raise FileNotFoundError(f'Images without GT JSON (not silently treated as background): {missing}')
    records = [(p, p.with_suffix('.json')) for p in images]
    return EvaluationDataset(records, class_names=CLASS_NAMES, label_aliases=LABEL_ALIASES, input_size=512, resize_size=1024,
        augment=False, random_crop=False, four_crop=True, cache_images=False)


In [ ]:
def predict_patches(dataset, checkpoint_path, device, ap_confidence=0.001, nms_iou=0.45, *, output_dir):
    """Resize to 1024², infer all four fixed 512² quadrants, including empty-GT crops."""
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False,
                            mmap=zipfile.is_zipfile(checkpoint_path))
    if checkpoint.get('num_classes') != 5 or checkpoint.get('architecture') != 'pdl1_flat_5class':
        raise ValueError('Expected a compatible 5-class detection checkpoint')
    names = checkpoint.get('class_names', checkpoint.get('params', {}).get('names'))
    if names is None or {int(k): v for k, v in names.items()} != CLASS_NAMES:
        raise ValueError('Checkpoint class order differs')
    with torch.no_grad():
        model = nn.yolo_v11_m(num_classes=5)
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    del checkpoint
    model = model.to(device).eval()
    results = PredictionStore(output_dir, create=True)
    try:
        with torch.inference_mode():
            for index in tqdm(range(len(dataset)), desc=Path(checkpoint_path).parent.name):
                image, labels, boxes, _ = dataset[index]
                xywh = boxes.numpy() * 512
                gt, keep = clip_boxes(np.concatenate(
                    [xywh[:, :2]-xywh[:, 2:]/2, xywh[:, :2]+xywh[:, 2:]/2], axis=1), 512, 512)
                with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                    raw = model(image.unsqueeze(0).to(device).float()/255)
                pred = util.non_max_suppression(raw.float(), confidence_threshold=ap_confidence,
                    iou_threshold=nms_iou, class_agnostic=True)[0].cpu().numpy()
                pred_boxes, valid = clip_boxes(pred[:, :4], 512, 512)
                source = str(dataset.records[index//4][0])
                results.append({'patch': f'{source}::q{index%4}', 'source_image': source,
                    'quadrant': index%4, 'gt_boxes': gt[keep],
                    'gt_labels': labels.numpy().astype(int)[keep], 'pred_boxes': pred_boxes[valid],
                    'pred_labels': pred[valid, 5].astype(int), 'pred_scores': pred[valid, 4]})
                del raw, pred, pred_boxes, image, labels, boxes, gt, xywh
    finally:
        del model
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    return results


In [ ]:
def calculate_dice(records, confidence=0.1):
    """Return union Dice, patch-level scores, and per-class pooled Dice."""
    rows = []
    for r in records:
        selected = np.asarray(r['pred_scores']) > confidence
        pred_boxes = np.asarray(r['pred_boxes'])[selected]
        pred_labels = np.asarray(r['pred_labels'])[selected]
        gt_boxes, gt_labels = np.asarray(r['gt_boxes']), np.asarray(r['gt_labels'])
        for class_id, class_name in [(-1, 'ALL')] + list(CLASS_NAMES.items()):
            gt = gt_boxes if class_id == -1 else gt_boxes[gt_labels == class_id]
            pred = pred_boxes if class_id == -1 else pred_boxes[pred_labels == class_id]
            rows.append({'patch': r['patch'], 'class_id': class_id, 'class_name': class_name,
                **rectangle_dice(gt, pred, 512, 512)})
    table = pd.DataFrame(rows)
    pooled_rows = []
    for (class_id, class_name), group in table.groupby(['class_id', 'class_name'], sort=False):
        denominator = group.gt_area.sum() + group.pred_area.sum()
        pooled_rows.append({'class_id': class_id, 'class_name': class_name,
            'dice': 2*group.intersection.sum()/denominator if denominator else float('nan')})
    pooled = pd.DataFrame(pooled_rows)
    union_dice = float(pooled.loc[pooled.class_id == -1, 'dice'].iloc[0])
    return union_dice, table, pooled[pooled.class_id != -1].reset_index(drop=True)


In [ ]:
def calculate_map(records):
    """Class-aware mAP@0.5 over all five classes, including other."""
    rows, score = average_precision_50(records, CLASS_NAMES)
    table = pd.DataFrame(rows)
    table['class_name'] = table.class_id.map(CLASS_NAMES)
    return score, table


In [ ]:
def calculate_pcc(records, confidence=0.1, thresholds=(1, 5, 10)):
    """Pearson correlation of patch CPS grades; report undefined CPS pairs explicitly."""
    rows = []
    for r in records:
        gt_counts = np.bincount(np.asarray(r['gt_labels'], dtype=int), minlength=5)
        selected = np.asarray(r['pred_scores']) > confidence
        pred_counts = np.bincount(np.asarray(r['pred_labels'], dtype=int)[selected], minlength=5)
        gt = cps_from_counts(gt_counts)['cps']
        pred = cps_from_counts(pred_counts)['cps']
        rows.append({'patch': r['patch'], 'gt_count': int(gt_counts.sum()),
            'pred_count': int(pred_counts.sum()), 'gt_cps': gt, 'pred_cps': pred,
            'gt_grade': cps_grade(gt, thresholds), 'pred_grade': cps_grade(pred, thresholds)})
    table = pd.DataFrame(rows)
    stats = paired_score_correlation(table.gt_grade, table.pred_grade)
    valid = np.isfinite(table.gt_grade) & np.isfinite(table.pred_grade)
    stats['grade_accuracy'] = float((table.loc[valid, 'gt_grade'] == table.loc[valid, 'pred_grade']).mean()) if valid.any() else float('nan')
    return stats, table


In [ ]:
dataset = load_test_folder(TEST_FOLDER)
print(f'{len(dataset.records)} images → {len(dataset)} fixed 512×512 patches | {DEVICE}')
for checkpoint in CHECKPOINTS.values():
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
if not 0 <= AP_CONFIDENCE <= CONFIDENCE <= 1:
    raise ValueError('Require 0 <= AP_CONFIDENCE <= CONFIDENCE <= 1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
pd.DataFrame(dataset.records, columns=['image', 'label']).to_csv(OUTPUT_DIR/'images.csv', index=False)
settings = {'class_names': CLASS_NAMES, 'label_aliases': LABEL_ALIASES, 'test_folder': str(TEST_FOLDER.resolve()), 'images': len(dataset.records),
    'patches': len(dataset), 'resize': 1024, 'crop': 512, 'random_crop': False,
    'confidence': CONFIDENCE, 'ap_confidence': AP_CONFIDENCE, 'nms_iou': NMS_IOU,
    'cps_grade_thresholds': [1, 5, 10], 'pcc_unit': '512x512 patch',
    'checkpoints': {name: str(path.resolve()) for name, path in CHECKPOINTS.items()}}
(OUTPUT_DIR/'settings.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2))


In [ ]:
summary_rows, class_tables = [], []
for name, checkpoint in CHECKPOINTS.items():
    predictions = predict_patches(dataset, checkpoint, DEVICE, AP_CONFIDENCE, NMS_IOU,
                                  output_dir=OUTPUT_DIR/f'{name}_patches')
    dice, dice_table, class_dice = calculate_dice(predictions, CONFIDENCE)
    map50, ap_table = calculate_map(predictions)
    pcc, cps_table = calculate_pcc(predictions, CONFIDENCE)
    per_class = class_dice.merge(ap_table, on=['class_id', 'class_name'], validate='one_to_one')
    per_class.insert(0, 'model', name)
    class_tables.append(per_class)
    per_class.to_csv(OUTPUT_DIR/f'{name}_metrics_per_class.csv', index=False)
    summary_rows.append({'model': name, 'patches': len(predictions), 'dice_union': dice,
        'dice_class_mean': class_dice.dice.mean(),
        'map50': map50, 'pcc_cps_grade': pcc['pcc'], 'grade_accuracy': pcc['grade_accuracy'],
        'pcc_valid_pairs': pcc['n_valid_pairs'], 'pcc_total_pairs': pcc['n_pairs'],
        'pcc_undefined_reason': pcc['pcc_undefined_reason']})
    dice_table.to_csv(OUTPUT_DIR/f'{name}_dice_per_patch.csv', index=False)
    ap_table.to_csv(OUTPUT_DIR/f'{name}_ap_per_class.csv', index=False)
    cps_table.to_csv(OUTPUT_DIR/f'{name}_cps_per_patch.csv', index=False)
    if SAVE_PREDICTIONS_JSON:
        save_predictions_json(predictions, OUTPUT_DIR/f'{name}_predictions.json')
    del predictions
    gc.collect()

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR/'summary.csv', index=False)
class_summary = pd.concat(class_tables, ignore_index=True)
class_summary.to_csv(OUTPUT_DIR/'metrics_per_class.csv', index=False)
# 클래스별 행과 전체 통합 행을 한 표에 표시합니다.
report_rows = []
for _, total in summary.iterrows():
    classes = class_summary[class_summary.model == total['model']]
    for _, row in classes.iterrows():
        report_rows.append({
            'model': total['model'], 'class': row['class_name'],
            'Dice (class / mean)': row['dice'], 'Dice (union)': np.nan,
            'AP@0.5 / mAP@0.5': row['ap50'],
            'CPS Grade PCC': np.nan, 'Grade accuracy': np.nan,
            'GT count': row['gt_count'], 'AP prediction count': row['ap_prediction_count'],
            'Patches': total['patches'], 'PCC valid pairs': np.nan, 'PCC total pairs': np.nan,
            'PCC undefined reason': '',
        })
    report_rows.append({
        'model': total['model'], 'class': '전체 (ALL)',
        'Dice (class / mean)': total['dice_class_mean'], 'Dice (union)': total['dice_union'],
        'AP@0.5 / mAP@0.5': total['map50'],
        'CPS Grade PCC': total['pcc_cps_grade'], 'Grade accuracy': total['grade_accuracy'],
        'GT count': classes.gt_count.sum(), 'AP prediction count': classes.ap_prediction_count.sum(),
        'Patches': total['patches'], 'PCC valid pairs': total['pcc_valid_pairs'],
        'PCC total pairs': total['pcc_total_pairs'],
        'PCC undefined reason': total['pcc_undefined_reason'],
    })
combined_report = pd.DataFrame(report_rows)
combined_report.to_csv(OUTPUT_DIR/'combined_metrics.csv', index=False)
print('클래스별 + 전체 통합 지표 (전체 행: 클래스 평균 Dice / mAP, PCC는 전체에만 적용)')
formats = {column: '{:.4f}' for column in
           ['Dice (class / mean)', 'Dice (union)', 'AP@0.5 / mAP@0.5', 'CPS Grade PCC', 'Grade accuracy']}
formats.update({column: '{:.0f}' for column in
                ['GT count', 'AP prediction count', 'Patches', 'PCC valid pairs', 'PCC total pairs']})
display(combined_report.style.format(formats, na_rep='—'))
print(f'Results: {OUTPUT_DIR.resolve()}')
